## Introduction

Understanding what employers expect in the analytics job market is not always straightforward, especially for students preparing to enter data-related roles. While there is a large amount of career advice available, it often lacks direct connection to actual hiring data. To address this gap, this project analyzes a dataset of 72,498 job postings from Lightcast, which includes information on job titles, required skills, salaries, and experience levels. Rather than relying on general assumptions, the analysis focuses on identifying patterns directly from observed job market data. 

The project is guided by three main questions: (1) What technical and transferable skills are most in demand for analytics-related roles in 2024?, (2) What factors — including experience, education, and location — are most strongly associated with salary, and (3) How can our team's current skill levels be benchmarked against market expectations to identify meaningful gaps?. To explore these questions, we apply a combination of exploratory data analysis, clustering, regression, and classification methods. The goal is not to provide definitive answers, but to develop a more grounded understanding of current labor market trends and their practical implications.



## Data Cleaning

This page explains how the Lightcast job-posting dataset was prepared for analysis. The goal is to create a clean and consistent dataset for exploratory analysis, skill comparisons, clustering, regression, and classification.

### Data Source

The project uses the file `lightcast_job_postings.csv`, stored in the `data` folder of the repository.

### Initial Dataset Overview

The dataset contains 131 variables and includes structured fields related to job titles, salary ranges, skills, work arrangements, education, experience, and occupational classification systems such as SOC, ONET, and NAICS.

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("data/lightcast_job_postings.csv", low_memory=False, on_bad_lines="skip")
df.shape

### Selected Columns

In [ ]:
selected_cols = [
    "TITLE_CLEAN",
    "LOT_OCCUPATION_NAME",
    "LOT_OCCUPATION_GROUP_NAME",
    "LOT_SPECIALIZED_OCCUPATION_NAME",
    "SALARY_FROM",
    "SALARY_TO",
    "ORIGINAL_PAY_PERIOD",
    "SKILLS_NAME",
    "SPECIALIZED_SKILLS_NAME",
    "COMMON_SKILLS_NAME",
    "SOFTWARE_SKILLS_NAME",
    "CITY_NAME",
    "STATE_NAME",
    "REMOTE_TYPE_NAME",
    "EDUCATION_LEVELS_NAME",
    "MIN_YEARS_EXPERIENCE",
    "MAX_YEARS_EXPERIENCE",
    "SOC_2021_3_NAME",
    "SOC_3_NAME",
    "ONET_NAME",
    "NAICS_2022_3_NAME",
    "NAICS3_NAME",
    "LIGHTCAST_SECTORS_NAME"
]

selected_cols = [c for c in selected_cols if c in df.columns]

df_clean = df[selected_cols].copy()
df_clean.shape

### Standardizing Core Variables

In [ ]:
df_clean["job_title"] = df_clean["TITLE_CLEAN"] if "TITLE_CLEAN" in df_clean.columns else np.nan
df_clean["occupation_name"] = df_clean["LOT_OCCUPATION_NAME"] if "LOT_OCCUPATION_NAME" in df_clean.columns else np.nan
df_clean["occupation_group"] = df_clean["LOT_OCCUPATION_GROUP_NAME"] if "LOT_OCCUPATION_GROUP_NAME" in df_clean.columns else np.nan
df_clean["specialized_occupation"] = df_clean["LOT_SPECIALIZED_OCCUPATION_NAME"] if "LOT_SPECIALIZED_OCCUPATION_NAME" in df_clean.columns else np.nan

df_clean["skills_all"] = df_clean["SKILLS_NAME"] if "SKILLS_NAME" in df_clean.columns else np.nan
df_clean["skills_specialized"] = df_clean["SPECIALIZED_SKILLS_NAME"] if "SPECIALIZED_SKILLS_NAME" in df_clean.columns else np.nan
df_clean["skills_common"] = df_clean["COMMON_SKILLS_NAME"] if "COMMON_SKILLS_NAME" in df_clean.columns else np.nan
df_clean["skills_software"] = df_clean["SOFTWARE_SKILLS_NAME"] if "SOFTWARE_SKILLS_NAME" in df_clean.columns else np.nan

df_clean["city"] = df_clean["CITY_NAME"] if "CITY_NAME" in df_clean.columns else np.nan
df_clean["state"] = df_clean["STATE_NAME"] if "STATE_NAME" in df_clean.columns else np.nan
df_clean["remote_type"] = df_clean["REMOTE_TYPE_NAME"] if "REMOTE_TYPE_NAME" in df_clean.columns else np.nan

df_clean["education_level"] = df_clean["EDUCATION_LEVELS_NAME"] if "EDUCATION_LEVELS_NAME" in df_clean.columns else np.nan
df_clean["min_experience"] = df_clean["MIN_YEARS_EXPERIENCE"] if "MIN_YEARS_EXPERIENCE" in df_clean.columns else np.nan
df_clean["max_experience"] = df_clean["MAX_YEARS_EXPERIENCE"] if "MAX_YEARS_EXPERIENCE" in df_clean.columns else np.nan

df_clean["soc_label"] = df_clean["SOC_2021_3_NAME"] if "SOC_2021_3_NAME" in df_clean.columns else df_clean["SOC_3_NAME"]
df_clean["naics_label"] = df_clean["NAICS_2022_3_NAME"] if "NAICS_2022_3_NAME" in df_clean.columns else df_clean["NAICS3_NAME"]
df_clean["onet_label"] = df_clean["ONET_NAME"] if "ONET_NAME" in df_clean.columns else np.nan
df_clean["sector_label"] = df_clean["LIGHTCAST_SECTORS_NAME"] if "LIGHTCAST_SECTORS_NAME" in df_clean.columns else np.nan

### Salary Preparation

In [ ]:
df_clean["salary_from_num"] = pd.to_numeric(df_clean["SALARY_FROM"], errors="coerce")
df_clean["salary_to_num"] = pd.to_numeric(df_clean["SALARY_TO"], errors="coerce")

df_clean["salary_mid"] = df_clean[["salary_from_num", "salary_to_num"]].mean(axis=1)

pay_period = df_clean["ORIGINAL_PAY_PERIOD"].astype(str).str.upper().str.strip()
annual_mask = pay_period.str.contains("YEAR|ANNUAL", na=False)

df_clean["annual_salary_mid"] = np.where(annual_mask, df_clean["salary_mid"], np.nan)

### Feature Engineering

In [ ]:
title_text = (
    df_clean["job_title"].fillna("").astype(str) + " " +
    df_clean["occupation_name"].fillna("").astype(str) + " " +
    df_clean["specialized_occupation"].fillna("").astype(str)
).str.lower()

analytics_pattern = (
    r"business analyst|data analyst|data scientist|data science|"
    r"machine learning|ml engineer|business intelligence|bi analyst|"
    r"analytics|data engineer|ai engineer|artificial intelligence"
)

df_clean["is_analytics_role"] = title_text.str.contains(analytics_pattern, regex=True, na=False).astype(int)

def count_items(x):
    if pd.isna(x):
        return 0
    parts = re.split(r"[;|]", str(x))
    parts = [p.strip() for p in parts if p and p.strip()]
    return len(parts)

for col in ["skills_all", "skills_specialized", "skills_common", "skills_software"]:
    df_clean[f"{col}_count"] = df_clean[col].apply(count_items)

df_clean["total_skill_count"] = (
    df_clean["skills_all_count"]
    + df_clean["skills_specialized_count"]
    + df_clean["skills_common_count"]
    + df_clean["skills_software_count"]
)

### Final Modeling Dataset

In [ ]:
final_cols = [
    "job_title",
    "occupation_name",
    "occupation_group",
    "specialized_occupation",
    "salary_from_num",
    "salary_to_num",
    "salary_mid",
    "annual_salary_mid",
    "remote_type",
    "city",
    "state",
    "education_level",
    "min_experience",
    "max_experience",
    "skills_all",
    "skills_specialized",
    "skills_common",
    "skills_software",
    "skills_all_count",
    "skills_specialized_count",
    "skills_common_count",
    "skills_software_count",
    "total_skill_count",
    "soc_label",
    "onet_label",
    "naics_label",
    "sector_label",
    "is_analytics_role"
]

df_model = df_clean[final_cols].copy()
df_model.shape

In [ ]:
df_model.to_csv("data/lightcast_cleaned.csv", index=False)

## Key Findings

- Python and SQL are the most in-demand skills  
- Salaries vary significantly by industry  
- Experience increases salary but with variability  

## Exploratory Data Analysis

This page summarizes the main descriptive patterns in the cleaned Lightcast job-posting dataset. The goal is to understand the distribution of analytics-related roles, salary patterns, work arrangements, and skills before applying clustering, regression, and classification methods.

### Load Cleaned Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_eda = pd.read_csv("data/lightcast_cleaned.csv")
df_eda.shape

### Analytics vs. Non-Analytics Roles

This section compares the number of postings classified as analytics-related versus all other roles.

In [ ]:
role_counts = df_eda["is_analytics_role"].value_counts(dropna=False)
role_counts

In [ ]:
role_counts.plot(kind="bar")
plt.title("Analytics vs. Non-Analytics Job Postings")
plt.xlabel("Is Analytics Role")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

### Salary Distribution

This section examines the distribution of annual salary midpoints for postings with annual salary information.

In [ ]:
salary_data = df_eda["annual_salary_mid"].dropna()
salary_data.describe()

In [ ]:
salary_data.plot(kind="hist", bins=30)
plt.title("Distribution of Annual Salary Midpoints")
plt.xlabel("Annual Salary Midpoint")
plt.ylabel("Frequency")
plt.show()

### Remote Work Patterns

This section summarizes the work arrangement categories available in the dataset.

In [ ]:
remote_counts = df_eda["remote_type"].value_counts(dropna=False)
remote_counts

In [ ]:
remote_counts.plot(kind="bar")
plt.title("Remote Work Arrangement Distribution")
plt.xlabel("Remote Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

### Education Requirements

This section shows the most common education levels requested in the postings.

In [ ]:
education_counts = df_eda["education_level"].value_counts(dropna=False).head(10)
education_counts

In [ ]:
education_counts.plot(kind="bar")
plt.title("Top Education Levels in Job Postings")
plt.xlabel("Education Level")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.show()

### Experience Requirements

This section summarizes years of experience requested across postings.

In [ ]:
df_eda[["min_experience", "max_experience"]].describe()

### Skill Count Distribution

This section examines how many skills are listed across job postings using the constructed skill count variable.

In [ ]:
df_eda["total_skill_count"].describe()

In [ ]:
df_eda["total_skill_count"].dropna().plot(kind="hist", bins=30)
plt.title("Distribution of Total Skill Counts")
plt.xlabel("Total Skill Count")
plt.ylabel("Frequency")
plt.show()

### Top Occupation Groups

This section highlights the most common occupation groups in the cleaned dataset.

In [ ]:
top_occ_groups = df_eda["occupation_group"].value_counts().head(10)
top_occ_groups

In [ ]:
top_occ_groups.plot(kind="bar")
plt.title("Top Occupation Groups")
plt.xlabel("Occupation Group")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.show()

### Key Takeaways

The exploratory analysis provides a first view of how analytics-related roles compare with the rest of the dataset. It also highlights salary availability, remote work patterns, education requirements, experience ranges, and variation in skill counts. These results help identify which variables are useful for skill comparisons, clustering, regression, and classification in the next sections.

## Modeling Methods

This page presents the main modeling methods used in the project. The goal is to identify patterns in analytics-related job postings and evaluate which factors help explain role groupings and salary differences. The methods include KMeans clustering, salary regression, and role classification.

### Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LinearRegression, LogisticRegression

df_ml = pd.read_csv("data/lightcast_cleaned.csv")
df_ml.shape

### Feature Selection Rationale

The selected variables reflect the main dimensions emphasized in the project: skills, experience, education, location, work arrangement, and occupational grouping. For salary analysis, numeric variables are first inspected to see how strongly they relate to annual salary.

In [ ]:
salary_corr = df_ml[
    ["annual_salary_mid", "salary_mid", "min_experience", "max_experience",
     "skills_all_count", "skills_specialized_count", "skills_common_count",
     "skills_software_count", "total_skill_count"]
].corr(numeric_only=True)["annual_salary_mid"].sort_values(ascending=False)

salary_corr

### KMeans Clustering

KMeans clustering is used to identify natural groupings of job postings based on numeric job characteristics. The clustering results are then compared with SOC labels to evaluate whether the clusters align with occupational categories.

In [ ]:
cluster_vars = [
    "salary_mid",
    "min_experience",
    "max_experience",
    "skills_all_count",
    "skills_specialized_count",
    "skills_common_count",
    "skills_software_count",
    "total_skill_count"
]

kmeans_df = df_ml[cluster_vars + ["soc_label"]].dropna().copy()
kmeans_df.shape

In [ ]:
X_cluster = kmeans_df[cluster_vars]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

silhouette_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    silhouette_scores[k] = silhouette_score(X_cluster_scaled, labels)

silhouette_scores

In [ ]:
best_k = max(silhouette_scores, key=silhouette_scores.get)
best_k

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_df["cluster"] = kmeans.fit_predict(X_cluster_scaled)

kmeans_df["cluster"].value_counts().sort_index()

In [ ]:
pca = PCA(n_components=2)
cluster_pca = pca.fit_transform(X_cluster_scaled)

plot_df = pd.DataFrame({
    "PC1": cluster_pca[:, 0],
    "PC2": cluster_pca[:, 1],
    "cluster": kmeans_df["cluster"]
})

plt.figure(figsize=(8, 6))
for c in sorted(plot_df["cluster"].unique()):
    subset = plot_df[plot_df["cluster"] == c]
    plt.scatter(subset["PC1"], subset["PC2"], label=f"Cluster {c}", alpha=0.5)

plt.title("KMeans Clusters of Job Postings")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.show()

In [ ]:
top_soc = kmeans_df["soc_label"].value_counts().head(8).index
cluster_soc_table = pd.crosstab(kmeans_df["cluster"], kmeans_df["soc_label"])[top_soc]
cluster_soc_table

### Regression Model

A regression model is used to estimate annual salary midpoint based on experience, skill counts, remote arrangement, education, location, and occupation group.

In [ ]:
salary_df = df_ml[df_ml["annual_salary_mid"].notna()].copy()
salary_df.shape

In [ ]:
reg_features = [
    "total_skill_count",
    "min_experience",
    "max_experience",
    "remote_type",
    "state",
    "education_level",
    "occupation_group"
]

X_reg = salary_df[reg_features]
y_reg = salary_df["annual_salary_mid"]

numeric_features = ["total_skill_count", "min_experience", "max_experience"]
categorical_features = ["remote_type", "state", "education_level", "occupation_group"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

reg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

reg_model.fit(X_train_reg, y_train_reg)
reg_pred = reg_model.predict(X_test_reg)

rmse = np.sqrt(mean_squared_error(y_test_reg, reg_pred))
r2 = r2_score(y_test_reg, reg_pred)

pd.DataFrame({
    "Metric": ["RMSE", "R2"],
    "Value": [rmse, r2]
})

### Classification Model

A classification model is used to predict whether a posting belongs to an analytics-related role using structured job-posting features.

In [ ]:
class_features = [
    "total_skill_count",
    "min_experience",
    "max_experience",
    "remote_type",
    "state",
    "education_level",
    "occupation_group",
    "sector_label"
]

X_clf = df_ml[class_features]
y_clf = df_ml["is_analytics_role"]

numeric_features_clf = ["total_skill_count", "min_experience", "max_experience"]
categorical_features_clf = ["remote_type", "state", "education_level", "occupation_group", "sector_label"]

preprocessor_clf = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features_clf),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features_clf)
])

clf_model = Pipeline([
    ("preprocessor", preprocessor_clf),
    ("model", LogisticRegression(max_iter=1000))
])

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

clf_model.fit(X_train_clf, y_train_clf)
clf_pred = clf_model.predict(X_test_clf)

acc = accuracy_score(y_test_clf, clf_pred)
f1 = f1_score(y_test_clf, clf_pred)

pd.DataFrame({
    "Metric": ["Accuracy", "F1 Score"],
    "Value": [acc, f1]
})

In [ ]:
cm = confusion_matrix(y_test_clf, clf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix: Analytics Role Classification")
plt.show()

### Interpretation for Job Seekers

The clustering results show whether job postings naturally group into distinct types and whether those groups resemble official occupational classifications. The regression model evaluates which job characteristics are associated with salary outcomes, while the classification model identifies which structured features best distinguish analytics-related roles. Together, these methods help translate labor market data into practical guidance for students and job seekers.



## NLP Methods

This section is included as an optional extension of the project. The main analysis in this project focuses on structured exploratory analysis, skill patterns, clustering, regression, and classification. Because the required evaluation is already addressed through those methods, natural language processing is treated as a secondary component in the current version of the project.

## Skill Gap Analysis

This page examines which skills appear most often in analytics-related job postings and what those patterns suggest for students preparing for business analytics, data science, and machine learning careers. The goal is to compare market demand with the types of skills that job seekers may need to strengthen.

### Load Cleaned Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_skills = pd.read_csv("data/lightcast_cleaned.csv")
df_skills.shape

### Focus on Analytics-Related Roles

This section limits the analysis to postings identified as analytics-related in the data cleaning stage.

In [ ]:
analytics_df = df_skills[df_skills["is_analytics_role"] == 1].copy()
analytics_df.shape

### Most Common Specialized Skills

This section identifies the most frequent specialized skills appearing in analytics-related roles.

In [ ]:
specialized = (
    analytics_df["skills_specialized"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

top_specialized = specialized.value_counts().head(15)
top_specialized

In [ ]:
top_specialized.plot(kind="bar")
plt.title("Top Specialized Skills in Analytics-Related Roles")
plt.xlabel("Skill")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.show()

### Most Common Software Skills

This section highlights the software tools that appear most often in analytics-related job postings.

In [ ]:
software = (
    analytics_df["skills_software"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

top_software = software.value_counts().head(15)
top_software

In [ ]:
top_software.plot(kind="bar")
plt.title("Top Software Skills in Analytics-Related Roles")
plt.xlabel("Software Skill")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.show()

### Most Common Common Skills

This section shows broader workplace and transferable skills that appear across analytics-related roles.

In [ ]:
common_skills = (
    analytics_df["skills_common"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

top_common = common_skills.value_counts().head(15)
top_common

In [ ]:
top_common.plot(kind="bar")
plt.title("Top Common Skills in Analytics-Related Roles")
plt.xlabel("Common Skill")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.show()

### Skill Counts by Role Type

This section compares the average total number of listed skills between analytics-related and non-analytics roles.

In [ ]:
skill_count_summary = (
    df_skills.groupby("is_analytics_role")["total_skill_count"]
    .mean()
    .reset_index()
)

skill_count_summary

In [ ]:
skill_count_summary.plot(
    x="is_analytics_role",
    y="total_skill_count",
    kind="bar",
    legend=False
)
plt.title("Average Total Skill Count by Role Type")
plt.xlabel("Is Analytics Role")
plt.ylabel("Average Total Skill Count")
plt.xticks(rotation=0)
plt.show()

### Practical Interpretation

The results on this page highlight which skills appear most often in analytics-related job postings. Technical tools and specialized skills help show what employers expect from candidates, while common skills reveal the broader workplace capabilities that still matter across roles. These patterns can help students identify where the strongest market demand exists and which areas may require additional preparation.

## Discussion

Looking at the results overall, several consistent patterns emerge. First, technical skills such as Python and SQL appear across a large share of job postings. Rather than acting as strong differentiators, these skills seem to function more as baseline requirements. In contrast, more advanced capabilities such as cloud computing and machine learning are more closely associated with higher-paying and more specialized roles. Second, experience plays a central role. Across different parts of the analysis, years of experience show a relatively strong relationship with salary. This suggests that practical, hands-on experience may be more influential than formal credentials alone. Third, there are signs that geographic constraints are becoming less important. Although many postings do not explicitly specify remote status, the presence of remote and hybrid roles suggests a shift toward a more flexible and competitive labor market. Finally, these findings are broadly consistent with existing research. For example, @mckinsey2023 suggests that automation may increase demand for high-skill roles, while @wef2023 highlights the importance of analytical thinking. The patterns observed in this dataset appear to align with these broader trends.

## Conclusion

This project analyzed job postings from 2024 to better understand current trends in the job analytics market. Several consistent takeaways emerge from the analysis. First, Python and SQL appear to be non-negotiable skills. They are present across a wide range of analytics roles, suggesting that they function as basic entry requirements rather than competitive advantages. Second, experience plays a critical role in salary outcomes. The results consistently show that years of hands-on experience are more strongly associated with compensation than formal credentials alone. Third, cloud computing and machine learning stand out as key skill gaps. These skills are more closely linked to higher-paying roles, indicating that they are important areas for future development. Fourth, the findings suggest that focused skill development is more effective than trying to learn a broad set of tools. Identifying a small number of high-value skills appears to be more beneficial than building a wide but shallow skill set. Finally, the skill gap identified in this project is specific and actionable. Rather than being abstract, it provides a clear direction for how students can improve their competitiveness in the job market. Overall, the results suggest that improving job prospects is less about doing more and more about focusing on the right areas, particularly by combining targeted skill development with meaningful practical experience.
